<img src='https://coderkian.com/wp-content/uploads/2020/04/Exercise-Fosters-Childs-Academic-Performance.jpg' width="1500" >

# <center> **Student Performance Data**
- The following data was obtained in a survey of students' math course in secondary school. 
- It contains a lot of interesting Features like social, gender and study information about students.

**Note:** For Visulization I've used **Plotly** (in case if it's not installed in your pc, just type **pip install plotly** in Cell) and **Seaborn** Library

#### Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly 
import plotly.express as ex
import seaborn as sns
from plotly.io import templates
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
from sklearn.preprocessing import LabelEncoder


sns.set(style= 'whitegrid', color_codes=True)
sns.set_theme(context='notebook',style='darkgrid',
              palette='deep',font='sans-serif',font_scale=1,color_codes=True,rc=None)
%matplotlib inline

In [ ]:
# to measure linear / non-linear relationship B/W two columns
# !pip install ppscore
# import ppscore as pps

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df= pd.read_csv('../input/student-performance-data/student_data.csv')
df.shape

In [ ]:
df.sample(4)

### 1. Visualizing the correlations between various features and grades and see which features have a significant impact on grades.
- Engineering the three grade parameters (G1, G2 and G3) as one feature for such comparisons.

In [ ]:
# AVG grade of each student
grad_mean= (df.G1 + df.G2 + df.G3) / 3
df['G_Mean'] = grad_mean
df.head(4)

In [ ]:
# Calculating the Predictive Power Score (PPS) matrix for all columns in the dataframe
# pps_matrix = pps.matrix(df)

In [ ]:
# pps_df = pd.DataFrame(pps_matrix)
# pps_df.loc[pps_df['x'] == 'G_Mean']

#### Using the Predictive Power Score, we can observe that School, Medu, Fedu, Pstatus, goout are the important features which impact our target column G_Mean

In [ ]:
plt.figure(figsize=(20,20))
sns.heatmap(df.corr(), vmin=-1, cmap="plasma_r", annot=True)
#same thing can be seen from the correlation as well

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot('sex', data = df, color='#00ddff', saturation=0.9)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x= 'sex', y = 'G_Mean', data = df, errwidth=3,saturation=1, palette='Blues_d') 
plt.legend(loc='upper left')

#### So we can observe there is not much impact of sex on the average grade column. It was also inferred from Predictive Power Score matrix

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Pstatus', data = df, palette='GnBu', saturation=0.9)

In [ ]:
# plt.figure(figsize=(1,1))
# sns.boxplot(x = 'Pstatus', y = 'G_Mean', hue='sex', data = df, saturation=1)
# plt.legend(loc='upper left')
# plt.ylabel('G Mean')
# plt.xlabel('P Status')
ex.box(x = 'Pstatus', y = 'G_Mean', color='sex', data_frame = df, template='seaborn', 
       notched=True, width=650, height=450)

#### With respect to Pstatus, we can observe that when Pstatus = A, male students are performing better than their female counterpart but when Pstatus = T, there is not much difference between the performance of male and female students.

In [ ]:
sns.countplot(x='Mjob', data = df, palette='rocket', saturation=0.9)
plt.show()
sns.countplot(x='Fjob', data = df, palette='dark', saturation=0.9)
plt.show()

In [ ]:
# plt.subplot(2,1,1)
# sns.boxplot(x=df['Mjob'], y = df['G_Mean'], palette='rocket', saturation=0.9)
ex.box(x=df['Mjob'], y = df['G_Mean'], color=df['Mjob'], height=330, width=800)
# plt.show()
# plt.subplot(2,1,1)
# sns.boxplot(x=df['Fjob'], y = df['G_Mean'], palette='rocket', saturation=0.9)
# plt.show()

In [ ]:
ex.box(x=df['Fjob'], y = df['G_Mean'], color=df['Fjob'], height=330, width=800)

#### Based on the Fjob, if Fjob = teacher, the average grade is a bit better. But in case of Mjob, when Mjob = health ie mother's are associated with health related servcies, the average grade is higher. But there also an outlier in case of Mjob = services where the grade goes down much below the average.

#### Otherwise Mjob and Fjob doesn't impact much on the average grade.

In [ ]:
sns.scatterplot(x = 'Medu', y = 'G_Mean', data = df)
plt.show()
sns.scatterplot(x = 'Fedu', y = 'G_Mean', data = df)
plt.show()

#### As the Medu (Mother Education) and Fedu (Father Education) increases, there is an increase in the average grade as well.

## <u> Que- 2:
- If there is a need for encoding some of the features, how would you go about it?
- can we consider combining certain encodings together ?

In [ ]:
df.select_dtypes(include='object').head(4)

 - There're 17 features having dtypes as object
- we've already seen from predictive pre processor matrix that school, pstatus, higher have more impact on the average grade
- so we can assign some ranking to these columns and the for the other columns, they are nominal data
- for the ordinal columns (in which Ranking is Important) we will go with the Label encoding and for the nominal column we will do One hot encoding

In [ ]:
encode_data = df.copy() # keeping a copy and not the same object reference so that we can use new_alcdata later on

In [ ]:
# Label Encoding the following columns as the student's average grade is better when these have appropriate value.
lencoder = LabelEncoder() 
encode_data['school'] = lencoder.fit_transform(encode_data['school'])
encode_data['higher'] = lencoder.fit_transform(encode_data['higher'])
encode_data['Pstatus'] = lencoder.fit_transform(encode_data['Pstatus'])

In [ ]:
# for the other categorical columns we'll do One hot encoding
for object_feature in encode_data.dtypes[encode_data.dtypes == 'object'].index:
     encode_data[object_feature] = encode_data[object_feature].astype('category')

for object_feature in encode_data.dtypes[encode_data.dtypes == 'category'].index:
        encode_dt = pd.get_dummies(encode_data[object_feature])
        encode_data.append(encode_dt)

In [ ]:
encode_data.info()

### 3.Figure out how family relation(famrel) and parents cohabitation(Pstatus) affect grades of students.

In [ ]:
# sns.boxplot(x = 'Pstatus', y = 'G_Mean', data = df)
ex.box(x = 'Pstatus', y = 'G_Mean', data_frame= df, color='Pstatus',height=350, width=510, title='G Mean Vs Pstatus')

In [ ]:
# sns.boxplot(x = 'famrel', y = 'G_Mean', hue='sex', data = df)
ex.box(x = 'famrel', y = 'G_Mean', data_frame= df, color='sex',height=350, width=510, title='G Mean Vs Famrel')

#### The Pstatus feature doesn't impact much on the grades of the student. But for famrel, there is an impact. The male students tends to perform better when the famrel is very low comapred to their female counterpart. When famrel >=3, both the male and female students perform at par with each other.

In [ ]:
data = {'Pstatus': encode_data.Pstatus, 'Famrel': encode_data.famrel, 'G_Mean': encode_data.G_Mean}
df_2 = pd.DataFrame(data, columns = ['Pstatus', 'Famrel', 'G_Mean'])
df_2.sample(4)

In [ ]:
plt.figure(figsize=(9,6))
sns.heatmap(df_2.corr(), vmin=-1, annot=True, cmap='YlGnBu_r')

### 4. Figure out which features in the data are skewed, and propose a way to remove skew from all such columns.
- skewness = 0: normally distributed
- skewness > 0: more weight in the left tail of the distribution(left skewed)
- skewness < 0: more weight in the right tail of the distribution(right skewed).

In [ ]:
skew_df = df.skew()
skew_df

In [ ]:
mean_absences = np.average(df.absences)
mean_absences

#### We can observe from skew matrix values that the feature 'absences' is left skewed. So we can apply log function on it to make it normally distributed. Before applying log function on it, we need to replace the the 0 values with the mean otherwise log function cannot be applied.

In [ ]:
check = df['absences'].replace(0, mean_absences)

In [ ]:
plt.figure(figsize=(12,5))
sns.distplot(np.log(check), kde = True, color='#0011ff', kde_kws={'color':'black'})
plt.show()

In [ ]:
ex.box(x= np.log10( np.log10(df.age) ), height=280, width=600)
# sns.boxplot(np.log10(np.log10(new_alcdata.age)))

In [ ]:
np.log(df['Dalc'])

#### In the Workday Alcohol Consumption(Dalc) feature also, it's left skewed and hence we apply log on it. But before apply for the second time, we replace the 0's generated by the previous log function with the mean of the log values generated earlier.

In [ ]:
plt.figure(figsize=(9,4))
sns.distplot(np.log(np.log(np.log(df['Dalc']).replace(0, np.mean(np.log(df['Dalc']))))), kde=True,
             color='#0011ff', kde_kws={'color':'black'})

#### From the skew matrix, it can be seen that that the 'famrel' feature is having negative skew that is it's right skewed. To remove the skewness from the data we apply the power function it.

In [ ]:
plt.figure(figsize=(11,5))
sns.distplot( np.power(df['famrel'],3), kde=True,
             color='#0011ff', kde_kws={'color':'black'} )

### Please do UpVote If You Liked the Notebook and Learned something new from the Notebook